# Data audit — Polish companies bankruptcy (`5year.arff`)

**Source:** UCI ML Repository, dataset 365 — Polish Companies Bankruptcy Data.

**Horizon.** File numbering is counter-intuitive: `Nyear.arff` holds financial ratios from
year *N* of the observation window and a bankruptcy label for *(5 − N)* years ahead.
`5year.arff` is therefore a **1-year-ahead** forecast — the strongest signal and the highest
positive rate of the five files.

**Purpose.** Verify that the data matches its documentation before any analysis, and record
the decisions that feed into preprocessing (stage 6) and feature engineering (stage 7).

**Headline findings**

1. The file is **pre-filtered**: companies with zero sales were removed before publication.
2. Class imbalance is severe — **6.9%** positives.
3. Missingness is **structural, not accidental** — it encodes the financial state of a company
   and is related to the target.

In [11]:
import numpy as np
import pandas as pd
from scipy.io.arff import loadarff

from src.config import RAW_DATA_DIR

ARFF_PATH = RAW_DATA_DIR / "5year.arff"

raw_data, meta = loadarff(ARFF_PATH)
df = pd.DataFrame(raw_data)

# scipy returns nominal attributes as bytes (b'0' / b'1'), not str.
# Comparisons against "0" would silently evaluate to False without decoding.
df["class"] = df["class"].str.decode("utf-8").astype(int)

## 1. Schema check

Does the file match its documented description: 5910 rows, 64 numeric features, one binary target?

In [12]:
non_numeric = [n for n, t in zip(meta.names(), meta.types()) if t != "numeric"]

print("shape:", df.shape)
print("non-numeric attributes:", non_numeric)
print("target declaration:", meta["class"])
print(df.dtypes.value_counts())

shape: (5910, 65)
non-numeric attributes: ['class']
target declaration: ('nominal', ('0', '1'))
float64    64
int64       1
Name: count, dtype: int64


## 2. Provenance: the data was pre-filtered

In [13]:
# The @relation header is truncated by scipy's parser, so read it from the file directly.
with open(ARFF_PATH, encoding="utf-8") as f:
    print(f.readline().strip())

@relation '5year-weka.filters.unsupervised.instance.SubsetByExpression-Enot ismissing(ATT20)'


The header records a Weka filter: `SubsetByExpression -E not ismissing(ATT20)`. Rows where
`Attr20` could not be computed were dropped before publication. Since
`Attr20 = (inventory × 365) / sales`, the ratio is undefined exactly when sales are zero —
so the removed rows are **companies with no revenue over the period**. The authors do not
document why this criterion was chosen; the mechanism is clear, the intent is not.

This is a **selection bias**: the sample was filtered on a property correlated with the outcome.
Companies with zero revenue — newly registered, dormant, or already ceasing operations — are
more likely to fail than average. Removing a group enriched in bankruptcies **deflates** the
observed positive rate, so the 6.9% below is lower than the true rate in the population of
Polish companies.

**Consequence for deployment:** the model is valid only for companies with non-zero revenue.
Applied to a dormant firm it will still return a probability, but nothing supports that number.
This goes in the README Limitations section.

## 3. Class balance

In [14]:
print(df["class"].value_counts())
print(df["class"].value_counts(normalize=True))

class
0    5500
1     410
Name: count, dtype: int64
class
0    0.930626
1    0.069374
Name: proportion, dtype: float64


410 positives out of 5910 — a positive rate of **0.069**.

This number is the **PR-AUC baseline**: a classifier assigning random scores achieves a PR-AUC
approximately equal to the positive rate. Every result later in the project is therefore reported
as a *lift over 0.069*, not as an absolute value.

**Accuracy is not used anywhere in this project.** A model that labels every company as healthy
is wrong 410 times and right 5500 times — 93.1% accuracy while catching no bankruptcies at all.
Accuracy weights errors by class frequency, but the rare class is the one that matters here, and
a missed bankruptcy and a false alarm do not carry the same cost. The metrics are PR-AUC and
precision@top-k (fixed in stage 3).

## 4. Missing values

In [15]:
na_rate = df.isna().mean().sort_values(ascending=False)

print(na_rate.head(10))
print("Attr20 missing rate:", na_rate["Attr20"])
print("rows with any NaN:", df.isna().any(axis=1).mean())
print(
    "rows with any NaN, excluding Attr37:",
    df.drop(columns="Attr37").isna().any(axis=1).mean(),
)

Attr37    0.431134
Attr27    0.066159
Attr45    0.045347
Attr60    0.045347
Attr24    0.022843
Attr64    0.018105
Attr53    0.018105
Attr28    0.018105
Attr54    0.018105
Attr21    0.017428
dtype: float64
Attr20 missing rate: 0.0
rows with any NaN: 0.4871404399323181
rows with any NaN, excluding Attr37: 0.15431472081218275


Missingness is dominated by a single column: `Attr37` at **43.1%**, six times the next one
(`Attr27`, 6.6%), with everything from the fifth position down below 2.5%. `Attr20` is at exactly
**0.0%**, confirming the filter above.

**Listwise deletion is not an option.** At least one feature is missing in **48.7%** of rows;
dropping them would destroy half the sample and roughly 200 of the 410 positives. Even excluding
`Attr37`, 15.4% of rows are still affected — missingness is spread thinly across many features,
not confined to one column.

Crucially, a missing value here does **not** mean incomplete data collection. Every feature is a
ratio, and a missing value marks an undefined denominator — a real financial state of the company.
The handling therefore cannot be a blanket fill.

## 5. Is missingness in `Attr37` related to the target?

`Attr37 = (current assets − inventories) / long-term liabilities`. The denominator is zero for
companies carrying **no long-term debt**, which makes the ratio undefined.

In [16]:
mask = df["Attr37"].isna()
print(df.groupby(mask)["class"].agg(["size", "sum", "mean"]))

        size  sum      mean
Attr37                     
False   3362  200  0.059488
True    2548  210  0.082418


Bankruptcy rate is **8.2%** among companies with `Attr37` missing versus **5.9%** among the rest —
a factor of 1.4. Both groups are large (2548 and 3362 rows, 210 and 200 positives), so the gap is
far outside sampling noise. **The fact of the value being missing carries signal in itself.**

A plausible economic reading: long-term credit has to be granted. A bank does not lend long to a
firm it considers risky, so the absence of long-term debt in an operating company can indicate
lack of access to credit rather than financial discipline — such a firm survives on short-term
obligations that must be rolled over continuously.

**Consequences:**

- **Linear branch:** imputation must be paired with a missing indicator
  (`SimpleImputer(add_indicator=True)`). Filling with the median alone erases the signal;
  filling with 0 or −1 is worse still, since it places "no debt at all" on the same numeric scale
  as real coverage ratios and forces one shared weight onto two different meanings.
- **Boosting branch:** leave NaN untouched. LightGBM learns a routing direction for missing values
  at every split, so it extracts this signal natively.

This is an **association, not a causal claim**. The elevated risk may come from the type of company
that tends to have no long-term debt rather than from the absence of debt itself. Sufficient for
prediction, not for causal conclusions — the same caveat applies to the SHAP results in stage 11.

## 6. Infinities

All features are ratios, so division by zero is possible. `isna()` does not catch `inf`: it is a
valid float that passes every missing-value check, then silently breaks scalers and linear models.

In [17]:
# np.isinf raises on non-numeric columns, so the target is excluded.
inf_mask = np.isinf(df.drop(columns="class"))

print("total inf values:", inf_mask.sum().sum())
print("affected rows:", inf_mask.any(axis=1).sum())

total inf values: 0
affected rows: 0


No infinities. Undefined ratios were encoded as missing values during data preparation; whether
`inf` appeared at an intermediate step cannot be recovered from the file, and ARFF has no syntax
for it in any case.

## 7. Constant and empty columns

`nunique()` ignores NaN: an all-NaN column returns 0, a true constant returns 1. Both are useless
to a model.

In [18]:
print(df.drop(columns="class").nunique().sort_values().head())

Attr59    3254
Attr37    3260
Attr6     3539
Attr21    4420
Attr9     4823
dtype: int64


Nothing to drop — the minimum is 3254 distinct values. The two lowest counts are explained rather
than anomalous: `Attr37` has 2548 missing values (excluded from the count), and `Attr59` has a
large spike at zero, examined next.

## 8. The same state encoded twice: `Attr37` missing and `Attr59 == 0`

`Attr59 = long-term liabilities / equity` places the same quantity in the **numerator**, whereas
`Attr37` has it in the denominator. Zero long-term debt therefore produces NaN in one feature and
a legitimate 0.0 in the other.

In [19]:
print(df["Attr59"].value_counts().head())
print("agreement:", (df["Attr37"].isna() == (df["Attr59"] == 0)).mean())

mismatch = df[df["Attr37"].isna() != (df["Attr59"] == 0)]
print(mismatch[["Attr37", "Attr59"]])

Attr59
0.000000    2547
0.100660       3
0.127770       3
0.002640       3
0.014242       2
Name: count, dtype: int64
agreement: 0.9991539763113367
      Attr37  Attr59
4586     0.0     0.0
4775     0.0     0.0
4852     NaN     NaN
4884     NaN     NaN
5880     NaN     NaN


The two encodings agree on **99.9%** of rows — 2547 zeros in `Attr59` against 2548 missing values
in `Attr37`. The five mismatches split into two distinct cases:

- **Rows 4852, 4884, 5880** — both features are NaN. These companies have zero equity as well, so
  `Attr59` breaks on its own denominator.
- **Rows 4586, 4775** — `Attr37` is 0.0 rather than NaN, so long-term debt does exist and the zero
  comes from the numerator (`current assets − inventories = 0`). `Attr59` is presumably rounded to
  zero at the dataset's stored precision.

**Implication for feature engineering.** The planned `has_no_long_term_debt` flag may be redundant
for LightGBM: the same partition is already reachable through a threshold on `Attr59`. For the
linear branch it is not redundant — there, `Attr59 = 0` sits on a continuous scale next to genuinely
small values and cannot be isolated as a category. Decided on CV with std in stage 7, not here.

## 9. Duplicate rows

Two questions: are the duplicates genuine repeats or an artifact of `NaN == NaN` matching, and do
identical feature vectors ever carry conflicting labels?

In [20]:
# Same keep= setting on both sides, otherwise the counts are not comparable.
print("duplicates incl. target:", df.duplicated().sum())
print("duplicates on features only:", df.drop(columns="class").duplicated().sum())

# keep=False marks every row of a duplicated group, not just the extra ones.
dups = df[df.duplicated(keep=False)]

print(dups.notna().sum(axis=1).describe())
print(dups["class"].value_counts())

duplicates incl. target: 60
duplicates on features only: 60
count    120.000000
mean      64.466667
std        0.888142
min       60.000000
25%       64.000000
50%       65.000000
75%       65.000000
max       65.000000
dtype: float64
class
0    116
1      4
Name: count, dtype: int64


**Decision: drop duplicates, keeping the first occurrence of each group.** Three findings support it:

1. **They are genuine repeats, not NaN artifacts.** Among the 120 rows involved, the least populated
   has 60 of 65 values filled and the median has all 65. Sixty float ratios matching to the last
   digit cannot happen by chance between two independent companies.
2. **Labels are consistent.** Duplicates counted with and without the target both give 60, so no
   pair carries contradictory labels.
3. **The cost is negligible.** Only 4 of the 120 rows are positives, so deduplication removes at
   most 2 of the 410 bankruptcies — under half a percent of the signal.

**What this prevents:** with copies of one company split across train and test, the model memorises
the object instead of the financial pattern, and the metric is inflated — a form of data leakage.

Deduplication is not a learned transformation (it estimates nothing from the data), so it belongs in
`data.py` at load time rather than inside the `Pipeline`.

## 10. Decisions carried forward

| Decision | Evidence |
|---|---|
| Drop duplicate rows at load time, `keep="first"` | 60 pairs, 60+ features matching, labels consistent, 4 positives involved |
| Do not drop rows with missing values | 48.7% of rows affected, ~200 of 410 positives would be lost |
| Never fill `Attr37` with 0 or −1 | Missing means "no long-term debt", a state, not a magnitude |
| Linear branch: median imputation **with** indicator | Missingness relates to the target (8.2% vs 5.9%) |
| Boosting branch: leave NaN untouched | LightGBM learns a split direction for missing values natively |
| Candidate feature `has_no_long_term_debt`, to be tested on CV | Possibly redundant for LightGBM given `Attr59` (99.9% agreement) |
| Metrics: PR-AUC and precision@top-k; accuracy excluded | 6.9% positives — a trivial all-negative model reaches 93.1% accuracy |
| Baseline PR-AUC level: **0.069** | Equal to the positive rate; all results reported as lift over it |
| README limitation: valid only for companies with non-zero revenue | Filter `not ismissing(ATT20)` removed zero-revenue firms |